### Ticking Clock
Wir wollen eine Animation einer Uhr erstellen, die die aktuelle Uhrzeit zeigt.  

In [ ]:
from datetime import datetime

now = datetime.now()
hours = now.hour
minutes = now.minute
seconds = now.second

hours, minutes, seconds

### Aufgabe
Vervollständige nachstehende Funktion und füge sie ins Modul `helpers` ein.

In [ ]:
def get_time():
    '''gibt ein Tuple (hour, minutes, seconds) zurueck'''
    ...

In [ ]:
import helpers as H
import time
from ipycanvas import hold_canvas


def draw_clock(canvas, center, radius):
    canvas.clear()
    H.draw_clockface(canvas, center, radius)
    H.draw_hands(canvas, center, radius)


def ticking_clock_1(canvas, center, radius):
    while True:
        with hold_canvas(canvas):
            canvas.clear()
            H.draw_clockface(canvas, center, radius)
            H.draw_hands(canvas, center, radius)
        time.sleep(1)

In [ ]:
import widget_helpers as W


canvas = W.get_canvas()
canvas

In [ ]:
ticking_clock_1(canvas, (50, 50), 40)

### Animation mit `asyncio`


Folgendes Pattern erstellt eine einfache Animation mit `asyncio`.
Eine solche Animation läuft im Hintergrund und blockiert das Notebook nicht.





```python
import asyncio


def animation(canvas, fps=25):
    # some initialization
  
    async def animate():
        while True:
            with hold_canvas(canvas):
                # draw frame
            await asyncio.sleep(1/fps)

    task = asyncio.create_task(animate(), name='my_animation')
    return task
```

Der Aufruf von `animate()` führt den Code im Funktionsbody nicht aus.
Statt dessen wird (ähnlich wie bei einer Generator-Funktion) eine
sog. Coroutine zurückzugeben.
Aus dieser Coroutine macht dann `asyncio.create_task(animate())` einen Task,
der in den Eventloop von Jupyterlab aufgenommen und gestartet wird.

Sobald `await asyncio.sleep(1/fps)` angetroffen wird,

```
draw frame 0
     ↓
await asyncio.sleep(1)
     ↓
(Eventloop kuemmert sich um andere Aufgaben)
     ↓
resume (nach 1 sec)
     ↓
draw frame 1
     ↓
...
```

Es ist wichtig, `task` zurückzugeben:
- Fehlermeldungen von `animate` werden nicht angezeigt.  
  `task.exception()` zeigt diese an.
- `task.cancel()` erlaubt, die Animation zu stoppen  

In [ ]:
import asyncio


def show_tasks():
    for i, task in enumerate(asyncio.all_tasks()):
        print(f'Task {i+1}: {task.get_name()}')
        print(f'  - Status:   {task._state}')
        print(f'  - Coroutine: {task.get_coro()}')
        print('-' * 40)


def cancel_task(task_name):
    for task in asyncio.all_tasks():
        if task.get_name() == task_name:
            print(f'trying to cancel task {task_name}')
            return task.cancel()

In [ ]:
show_tasks()

In [ ]:
import helpers as H
from vector import Vector as Vec
from ipycanvas import hold_canvas


def ticking_clock(canvas, center, radius):
    r = 1.2 * radius
    ur = Vec(*center) - Vec(r, r)

    async def animate():
        while True:
            with hold_canvas(canvas):
                canvas.clear_rect(ur.x, ur.y, 2*r)
                H.draw_clockface(canvas, center, radius)
                H.draw_hands(canvas, center, radius)
            await asyncio.sleep(1)

    task = asyncio.create_task(animate(), name='ticking_clock')
    return task

In [ ]:
import widget_helpers as W


canvas = W.get_canvas()
canvas

In [ ]:
radius = 40
center = (50, 50)
task = ticking_clock(canvas, center, radius)

In [ ]:
task

In [ ]:
task.exception()

In [ ]:
task.cancel()

In [ ]:
show_tasks()